# Tugas 7 Pencarian dokumen 

#### Rafly Faldiansyah Putra 210411100063

# Transformasi SVD pada TF-IDF

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
import re
from IPython.display import display, Markdown

# 1. Membaca Dataset
# Membaca file CSV yang berisi dataset
df = pd.read_csv("preprocessing-kompas.csv")

# Mengganti nilai NaN dengan string kosong agar tidak ada data kosong yang menyebabkan error
df['stopword_removal'] = df['stopword_removal'].fillna('')

# 2. Mengonversi Data Teks ke TF-IDF
# Menginisialisasi TfidfVectorizer dengan normalisasi L2
vectorizer = TfidfVectorizer(norm='l2')

# Menghitung nilai TF-IDF untuk setiap dokumen
tfidf_matrix = vectorizer.fit_transform(df['stopword_removal'])

# Mengubah hasil TF-IDF menjadi DataFrame untuk kemudahan akses
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

# 3. Reduksi Dimensi dengan Truncated SVD
# Menginisialisasi TruncatedSVD untuk mereduksi dimensi menjadi 100 komponen (atau sesuai kebutuhan)
svd = TruncatedSVD(n_components=100, random_state=42)

# Mengaplikasikan SVD pada matriks TF-IDF untuk menghasilkan matriks dengan dimensi yang lebih rendah
svd_matrix = svd.fit_transform(tfidf_matrix)

# Menyimpan hasil SVD dalam DataFrame untuk digunakan dalam perhitungan kemiripan
svd_df = pd.DataFrame(svd_matrix)

# Menampilkan 10 baris pertama dari DataFrame hasil SVD
svd_df.head(1000)



,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.318048,0.336699,0.001047,0.008270,-0.257332,0.017326,0.159126,-0.324972,-0.082744,0.175946,...,0.025443,0.032022,0.003989,-0.084826,0.001480,0.006058,0.009300,0.029638,0.005525,0.003858
1,0.267889,0.265209,0.001206,-0.001144,-0.145906,0.051709,-0.002118,-0.098570,0.018178,-0.014501,...,-0.071177,-0.047667,-0.032383,0.038653,0.032659,0.012287,-0.018055,0.015435,0.003373,0.001066
2,0.275993,0.289724,0.009255,0.018454,-0.282478,0.025928,0.141696,-0.350281,-0.059327,0.174348,...,0.023016,0.017567,-0.023269,-0.001327,0.025918,-0.024781,0.000734,0.035208,-0.000262,0.001894
3,0.184904,-0.045614,-0.038867,0.105367,-0.001282,-0.128633,0.080001,0.046412,-0.074667,-0.001831,...,0.010405,-0.014090,-0.002174,-0.009361,-0.001838,-0.003785,0.006885,-0.008148,-0.002289,-0.001875
4,0.336624,0.377235,0.005007,-0.022246,-0.212861,0.020914,0.119120,-0.363587,-0.089158,0.186927,...,-0.040972,-0.035931,0.042560,0.074858,0.003522,0.046988,-0.006124,-0.046595,-0.007314,0.000503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,0.135096,-0.082512,0.057765,-0.012827,0.060773,0.104402,-0.004441,0.079390,0.519378,0.481841,...,-0.004262,-0.002475,-0.001884,-0.054075,-0.008057,-0.005671,-0.015422,-0.000973,0.020449,0.001359
96,0.136410,-0.027847,0.022742,0.016475,-0.004543,0.027487,0.100443,0.050808,0.128822,-0.046311,...,-0.022888,-0.001423,-0.003475,-0.012415,-0.011740,-0.006815,0.001626,0.000196,0.008629,-0.003298
97,0.089245,-0.069275,-0.004827,0.022257,0.011292,-0.018394,0.021997,0.033593,0.044537,-0.002819,...,0.009351,-0.002837,-0.000367,0.014356,0.008336,0.004960,-0.010100,0.001605,0.000423,-0.002849
98,0.127469,-0.039437,0.019895,-0.025436,0.002352,0.022569,-0.000966,0.033656,0.097731,0.022922,...,-0.039153,-0.026095,0.048038,0.021043,-0.016784,0.014634,0.006370,0.006212,0.004308,-0.006296


# Mencari Dokumen dengan Cosine Similiarity

In [2]:
# Fungsi untuk mencari dokumen terkait
def search_documents(query, top_n=100, similarity_threshold=0.5):
    # 4. Menghitung TF-IDF untuk kalimat inputan
    query_tfidf = vectorizer.transform([query])
    
    # 5. Mengurangi dimensi kalimat inputan
    query_svd = svd.transform(query_tfidf)
    
    # 6. Menghitung kemiripan kosinus
    cosine_similarities = cosine_similarity(query_svd, svd_df)
    
    # 7. Menyaring hasil kemiripan berdasarkan threshold
    similar_indices = [
        idx for idx, score in enumerate(cosine_similarities[0])
        if score >= similarity_threshold
    ]
    
    # 8. Mengambil dokumen terkait berdasarkan threshold
    related_docs = df.iloc[similar_indices]
    
    # Membatasi jumlah dokumen yang diambil
    return related_docs.head(top_n)

# Contoh penggunaan - Input dari pengguna
input_query = input("Masukkan kalimat pencarian Anda: ")
related_documents = search_documents(input_query)

# Menampilkan dokumen terkait dengan detail yang diminta
print("=" * 50)
print(f"{'Dokumen Terkait':^50}")
print("=" * 50)
displayed_count = 0

for index, row in related_documents.iterrows():
    print(f"Dokumen #{displayed_count + 1}")
    print(f"Judul    : {row['judul']}")
    print(f"Tanggal  : {row['tanggal']}")
    print(f"Kategori : {row['kategori']}")
    print(f"Isi Berita:\n{row['isi_berita']}")
    print("-" * 50)  # Pemisah antar dokumen
    displayed_count += 1

# Menampilkan jumlah dokumen yang ditampilkan
print(f"\nJumlah dokumen yang ditampilkan: {displayed_count}")
print("=" * 50)


Masukkan kalimat pencarian Anda:  Prabowo


                 Dokumen Terkait                  
Dokumen #1
Judul    : Wamendagri Sebut Prabowo Ingin Perbaiki Sistem Politik Berbiaya Sangat Mahal
Tanggal  : 19/11/2024, 12:31 WIB
Kategori : Politik
Isi Berita:
JAKARTA, KOMPAS.com - Wakil Menteri Dalam Negeri (Wamendagri) Bima Arya mengatakan, Presiden Prabowo Subianto ingin memperbaiki sistem politik yang saat ini dinilai berbiaya mahal. Keinginan Prabowo ini diucapkan langsung saat Bima Arya dipanggil ke Kertanegara sebagai calon Wamendagri. "Yang pertama kali dia sampaikan adalah 'tolong Kemendagri lakukan kajian tentang sistem pemiliu kita, tidak efektif, tidak efisien,' kira-kira begitu," ujar Bima dalam acara diskusi di Akmani Hotel, Jakarta Pusat, Selasa (19/11/2024). Bima Arya kemudian menegaskan, Prabowo menangkap keresahan masyarakat terkait biaya politik yang mahal. Baca juga: Inovasi Mobil Keliling Dukcapil Surakarta Tuai Pujian dari Wamendagri Bima Arya Selain itu, Prabowo juga disebut ingin agar pemilu bisa mempersatuk

In [5]:
# Contoh penggunaan - Input dari pengguna
input_query = "sepak bola"
related_documents = search_documents(input_query)

# Menampilkan dokumen terkait dengan detail yang diminta
print("=" * 50)
print(f"{'Dokumen Terkait':^50}")
print("=" * 50)
displayed_count = 0

for index, row in related_documents.iterrows():
    print(f"Dokumen #{displayed_count + 1}")
    print(f"Judul    : {row['judul']}")
    print(f"Tanggal  : {row['tanggal']}")
    print(f"Kategori : {row['kategori']}")
    print(f"Isi Berita:\n{row['isi_berita']}")
    print("-" * 50)  # Pemisah antar dokumen
    displayed_count += 1

# Menampilkan jumlah dokumen yang ditampilkan
print(f"\nJumlah dokumen yang ditampilkan: {displayed_count}")
print("=" * 50)

                 Dokumen Terkait                  
Dokumen #1
Judul    : Road to Dender: Mewujudkan Mimpi Talenta Muda Indonesia
Tanggal  : 23/11/2024, 05:00 WIB
Kategori : Olahraga
Isi Berita:
KOMPAS.com - Program Road to Dender yang digagas oleh Pemilik FCV Dender, Sihar Sitorus, bertujuan besar menghubungkan Belgia dan Indonesia melalui sepak bola.  Program ini masih dalam tahap perencanaan, tetapi fokusnya adalah menemukan dan mengembangkan talenta muda berbakat di Indonesia. “Kami ingin menscouting talenta muda dari berbagai daerah di Indonesia, baik dari bagian barat maupun timur. Untuk usia, kami memprioritaskan pemain muda dari kategori U-8 hingga U-16, karena usia tersebut merupakan golden years dalam pengembangan bakat,” ujar Sihar Sitorus kepada wartawan di Jakarta pada Jumat (22/11/2024). Baca juga: Tersisa 4 Pertandingan, Kapan Timnas Indonesia Bertanding Lagi? Berikut Jadwal Lengkapnya Sihar menegaskan, program ini dirancang untuk menjadi langkah solid dalam pembangunan s